# 07 - Reaction-Diffusion Equation

Solve $u_t = 0.01 u_{xx} + u(1-u)$

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, sys
sys.path.insert(0, '../src')
from pinns import MLP, set_seed
from pinns.utils.derivatives import gradient
set_seed(42)
D = 0.01
model = MLP(input_dim=2, output_dim=1, hidden_dims=[32,32,32], activation='tanh')

In [ ]:
def pde_loss(model, x, t):
    xt = torch.cat([x, t], dim=1).requires_grad_(True)
    u = model(xt)
    g = gradient(u, xt)
    u_t, u_x = g[:, 1:2], g[:, 0:1]
    u_xx = gradient(u_x, xt)[:, 0:1]
    return torch.mean((u_t - D*u_xx - u*(1-u))**2)
def bc_loss(model, n=50):
    t = torch.rand(n, 1)*2
    return torch.mean(model(torch.cat([torch.zeros(n,1), t], dim=1))**2 + model(torch.cat([torch.ones(n,1), t], dim=1))**2)
def ic_loss(model, n=50):
    x = torch.rand(n, 1)
    u0 = model(torch.cat([x, torch.zeros(n,1)], dim=1))
    return torch.mean((u0 - torch.where(x<0.5, torch.ones_like(x), torch.zeros_like(x)))**2)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(1500):
    opt.zero_grad()
    loss = pde_loss(model, torch.rand(100,1), torch.rand(100,1)*2) + 10*bc_loss(model) + 10*ic_loss(model)
    loss.backward(); opt.step()
    if (epoch+1) % 500 == 0: print(f'Epoch {epoch+1}: {loss.item():.4e}')

In [ ]:
X, T = torch.meshgrid(torch.linspace(0,1,80), torch.linspace(0,2,80), indexing='ij')
xt = torch.stack([X.flatten(), T.flatten()], dim=1)
with torch.no_grad(): u = model(xt).reshape(80, 80)
plt.figure(figsize=(10,6))
plt.contourf(X.numpy(), T.numpy(), u.numpy(), levels=50, cmap='viridis')
plt.colorbar(label='u')
plt.xlabel('x'); plt.ylabel('t'); plt.title('Reaction-Diffusion')
plt.tight_layout(); plt.show()